In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# For nice plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ All libraries imported successfully!")

In [ ]:
# Load the dataset
df = pd.read_csv('chilledwater.csv', parse_dates=['timestamp'])

print(f"Dataset loaded!")
print(f"Shape: {df.shape} (rows, columns)")
print(f"\nTime range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"\nFirst few columns: {df.columns[:5].tolist()}")

In [ ]:
# Step 1: Remove columns with >95% missing data
missing_pct = df.isnull().sum() / len(df)
cols_to_keep = missing_pct[missing_pct < 0.95].index
df_clean = df[cols_to_keep].copy()

print(f"After removing empty columns: {df_clean.shape}")

# Step 2: Identify building columns (everything except timestamp)
building_cols = [c for c in df_clean.columns if c != 'timestamp']

# Step 3: Select top 5 buildings with most complete data
completeness = df_clean[building_cols].count().sort_values(ascending=False)
top_5_buildings = completeness.head(5).index.tolist()

print(f"\nTop 5 buildings selected:")
for i, building in enumerate(top_5_buildings, 1):
    print(f"{i}. {building}")

In [ ]:
# Work with just these 5 buildings for the demo
mvp_buildings = top_5_buildings

# Create working dataframe
df_work = df_clean[['timestamp'] + mvp_buildings].copy()
df_work.set_index('timestamp', inplace=True)

# Handle missing values
df_work = df_work.interpolate(method='time', limit=24)  # Fill gaps up to 24 hours
df_work = df_work.fillna(method='ffill', limit=24)      # Forward fill remaining

print("Data cleaned!")
print(df_work.head())
print(f"\nMissing values remaining: {df_work.isnull().sum().sum()}")

In [ ]:
# Plot all 5 buildings
plt.figure(figsize=(15, 8))
for col in df_work.columns:
    plt.plot(df_work.index, df_work[col], label=col.split('_')[-1], alpha=0.7)

plt.title('Chilled Water Consumption - 5 Buildings (2 Years)', fontsize=16)
plt.xlabel('Date')
plt.ylabel('Consumption')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

print("📊 This shows the baseline consumption patterns")

In [ ]:
# Create features for ML model
def create_features(df):
    features = pd.DataFrame(index=df.index)
    
    # Time features
    features['hour'] = df.index.hour
    features['day_of_week'] = df.index.dayofweek
    features['month'] = df.index.month
    
    # Statistical features for each building
    for col in df.columns:
        # Rolling statistics
        features[f'{col}_roll_mean_24h'] = df[col].rolling(24, min_periods=1).mean()
        features[f'{col}_roll_std_24h'] = df[col].rolling(24, min_periods=1).std()
        features[f'{col}_diff'] = df[col].diff()
        
        # Lag features (what was consumption 1 hour ago, 24 hours ago)
        features[f'{col}_lag_1h'] = df[col].shift(1)
        features[f'{col}_lag_24h'] = df[col].shift(24)
    
    return features.fillna(0)

features = create_features(df_work)
print(f"Features created! Shape: {features.shape}")
print(f"\nFeature columns: {features.columns[:5].tolist()}... (and {len(features.columns)-5} more)")

In [ ]:
# TIER 1: IMPROVED Statistical Detection (Rolling Z-Score)
def statistical_detection(df, window=168, threshold=3.5):  # 168 = 1 week window
    """
    Use rolling Z-score instead of global Z-score.
    This prevents flagging all summer data as anomalous.
    """
    anomalies = pd.DataFrame(index=df.index)
    
    for col in df.columns:
        # Calculate rolling mean and std (7-day window)
        rolling_mean = df[col].rolling(window=window, min_periods=24).mean()
        rolling_std = df[col].rolling(window=window, min_periods=24).std()
        
        # Z-score based on recent history, not entire dataset
        z_scores = np.abs((df[col] - rolling_mean) / rolling_std)
        
        # Only flag if deviation is extreme AND usage is above recent average
        # (We don't care about "too low" usage, only suspicious spikes)
        is_high = df[col] > rolling_mean * 1.5  # 50% above recent average
        anomalies[f'{col}_stat'] = (z_scores > threshold) & is_high
    
    return anomalies

stat_anomalies = statistical_detection(df_work)

print("✅ Improved statistical detection complete!")
print("Using 7-day rolling windows (prevents seasonal false positives)\n")

for col in df_work.columns:
    count = stat_anomalies[f'{col}_stat'].sum()
    pct = (count / len(df_work)) * 100
    print(f"  {col.split('_')[-1]:12} | {count:4} anomalies ({pct:5.2f}%)")

In [ ]:
# TIER 2: IMPROVED ML Detection
def ml_detection(df, features, contamination=0.02):  # Lower contamination (2%)
    """
    Lower contamination = fewer, more certain anomalies detected
    """
    # Use more relevant features
    feature_cols = [c for c in features.columns 
                   if any(x in c for x in ['roll_mean', 'diff', 'lag_24h'])]
    
    X = features[feature_cols].fillna(0)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Isolation Forest with stricter settings
    iso_forest = IsolationForest(
        contamination=contamination, 
        random_state=42, 
        n_estimators=100,
        max_samples='auto',
        max_features=0.8  # Use subset of features for each tree
    )
    
    predictions = iso_forest.fit_predict(X_scaled)
    return predictions == -1

ml_anomalies = ml_detection(df_work, features)

print(f"✅ ML Detection complete!")
print(f"Total ML anomalies: {ml_anomalies.sum()} ({ml_anomalies.mean()*100:.2f}% of data)")
print("(Should be 1-5% for realistic anomaly detection)")

In [ ]:
# TIER 2B: Local Outlier Factor (LOF) Detection
from sklearn.neighbors import LocalOutlierFactor

def lof_detection(df, features, contamination=0.02):
    """
    Local Outlier Factor: detects anomalies based on local density deviation.
    Good at finding contextual/local anomalies that Isolation Forest may miss.
    """
    feature_cols = [c for c in features.columns 
                   if any(x in c for x in ['roll_mean', 'diff', 'lag_24h'])]
    
    X = features[feature_cols].fillna(0)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    n_neighbors = min(20, len(X_scaled) - 1)
    lof = LocalOutlierFactor(
        n_neighbors=n_neighbors,
        contamination=contamination,
        novelty=False
    )
    predictions = lof.fit_predict(X_scaled)
    return predictions == -1

lof_anomalies = lof_detection(df_work, features)

print(f"✅ LOF Detection complete!")
print(f"Total LOF anomalies: {lof_anomalies.sum()} ({lof_anomalies.mean()*100:.2f}% of data)")


In [ ]:
# TIER 2C: One-Class SVM Detection
from sklearn.svm import OneClassSVM

def ocsvm_detection(df, features, nu=0.02):
    """
    One-Class SVM: learns a tight boundary around normal data.
    Effective for non-linear anomaly patterns. nu ≈ contamination rate.
    """
    feature_cols = [c for c in features.columns 
                   if any(x in c for x in ['roll_mean', 'diff', 'lag_24h'])]
    
    X = features[feature_cols].fillna(0)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    ocsvm = OneClassSVM(
        kernel='rbf',
        nu=nu,
        gamma='scale'
    )
    predictions = ocsvm.fit_predict(X_scaled)
    return predictions == -1

ocsvm_anomalies = ocsvm_detection(df_work, features)

print(f"✅ One-Class SVM Detection complete!")
print(f"Total OCSVM anomalies: {ocsvm_anomalies.sum()} ({ocsvm_anomalies.mean()*100:.2f}% of data)")


In [ ]:
# TIER 3: IMPROVED Temporal Detection (Account for Seasonality)
def temporal_detection(df, threshold=3.0):
    """
    Compare against same hour AND same month (prevents flagging all summer data)
    """
    anomalies = pd.DataFrame(index=df.index)
    
    for col in df.columns:
        # Create seasonal baseline (same hour, same month)
        seasonal_baseline = df.groupby([df.index.month, df.index.hour])[col].median()
        
        # Get expected value for each timestamp
        expected = []
        for ts in df.index:
            try:
                expected.append(seasonal_baseline[(ts.month, ts.hour)])
            except KeyError:
                expected.append(df[col].median())  # Fallback
        
        expected = pd.Series(expected, index=df.index)
        
        # Calculate deviation from seasonal expectation
        deviation = np.abs(df[col] - expected) / expected
        
        # Only flag if:
        # 1. Deviation is > 300% (3x normal for that time), AND
        # 2. Absolute value is significantly above recent minimum
        
        recent_min = df[col].rolling(168, min_periods=24).min()
        is_significant = df[col] > (recent_min * 2)  # At least 2x the minimum
        
        anomalies[f'{col}_temp'] = (deviation > threshold) & is_significant
    
    return anomalies

temp_anomalies = temporal_detection(df_work)

print("✅ Temporal detection complete! (Seasonal-adjusted)")
for col in df_work.columns:
    count = temp_anomalies[f'{col}_temp'].sum()
    print(f"  {col.split('_')[-1]:12} | {count:4} seasonal anomalies")

In [ ]:
# === MODEL PERFORMANCE COMPARISON ===
# Compare Isolation Forest, LOF, and One-Class SVM
# Metrics: Classification + Regression style evaluation against consensus ground truth

from sklearn.metrics import (
    confusion_matrix
)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ----- Build a pseudo ground-truth using majority voting across all 5 signals -----
# (Rolling Z-Score + Temporal + Isolation Forest + LOF + One-Class SVM)
# A point is "true anomaly" if at least 3 of 5 methods agree
all_votes = (
    stat_anomalies[[c for c in stat_anomalies.columns]].any(axis=1).astype(int) +
    temp_anomalies[[c for c in temp_anomalies.columns]].any(axis=1).astype(int) +
    ml_anomalies.astype(int) +
    lof_anomalies.astype(int) +
    ocsvm_anomalies.astype(int)
)
consensus_gt = (all_votes >= 3).astype(int).values  # Pseudo ground truth (0/1)

models = {
    "Isolation Forest": ml_anomalies,
    "Local Outlier Factor": lof_anomalies,
    "One-Class SVM": ocsvm_anomalies,
}

# ===== COMPUTE ALL METRICS =====
rows = []
for name, preds in models.items():
    p = preds.values.astype(int) if hasattr(preds, 'values') else preds.astype(int)
    y_true = consensus_gt
    y_pred = p

    # Confusion matrix components
    tp = ((y_pred == 1) & (y_true == 1)).sum()
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    fn = ((y_pred == 0) & (y_true == 1)).sum()
    tn = ((y_pred == 0) & (y_true == 0)).sum()

    # Classification metrics
    precision   = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall      = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    accuracy = (tp + tn) / len(p)
    anomaly_rate = y_pred.mean() * 100

    # # Balanced Accuracy (average of recall and specificity)
    # bal_acc = balanced_accuracy_score(y_true, y_pred)

    # # Matthews Correlation Coefficient (best single metric for imbalanced data)
    # mcc = matthews_corrcoef(y_true, y_pred)

    # # Cohen's Kappa (inter-rater agreement)
    # kappa = cohen_kappa_score(y_true, y_pred)

    # # Jaccard Score (intersection over union)
    # jaccard = jaccard_score(y_true, y_pred)

    # # Regression-style metrics (treating binary predictions as continuous 0/1)
    # mse_val = mean_squared_error(y_true, y_pred)
    # mae_val = mean_absolute_error(y_true, y_pred)
    # r2_val  = r2_score(y_true, y_pred)

    rows.append({
        "Model": name,
        "Anomalies": int(y_pred.sum()),
        "Anomaly Rate (%)": round(anomaly_rate, 2),
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1-Score": round(f1, 4),
        # "Specificity": round(specificity, 4),
        "Accuracy": round(accuracy, 4),
        # "Balanced Acc.": round(bal_acc, 4),
        # "MCC": round(mcc, 4),
        # "Cohen Kappa": round(kappa, 4),
        # "Jaccard": round(jaccard, 4),
        # "MSE": round(mse_val, 6),
        # "MAE": round(mae_val, 6),
        # "R2 Score": round(r2_val, 4),
    })

perf_df = pd.DataFrame(rows).set_index("Model")

# ===== PRINT RESULTS =====
print("=" * 80)
print("   MODEL PERFORMANCE COMPARISON TABLE (vs. Consensus Ground Truth)")
print("=" * 80)
print()

# Split into sections for readability
print("--- Classification Metrics ---")
class_cols = ["Anomalies", "Anomaly Rate (%)", "Precision", "Recall", "F1-Score",
               "Accuracy"]
print(perf_df[class_cols].to_string())
print()

# # print("--- Advanced / Agreement Metrics ---")
# # adv_cols = ["MCC", "Cohen Kappa", "Jaccard"]
# # print(perf_df[adv_cols].to_string())
# # print()

# # print("--- Regression-style Metrics (binary 0/1 predictions vs ground truth) ---")
# # reg_cols = ["MSE", "MAE", "R2 Score"]
# # print(perf_df[reg_cols].to_string())
# # print()

# # ===== VISUALIZATION: Classification Metrics Bar Chart =====
# fig, axes = plt.subplots(2, 3, figsize=(18, 10))
# fig.suptitle("ML Model Performance Comparison for Anomaly Detection",
#              fontsize=16, fontweight="bold", y=1.02)

# metrics_to_plot = ["Precision", "Recall", "F1-Score", "MCC", "Cohen Kappa", "Jaccard"]
# colors = ["#1E88E5", "#43A047", "#E53935", "#8E24AA", "#FB8C00", "#00ACC1"]
# model_names = perf_df.index.tolist()
# bar_width = 0.5

# for ax, metric, color in zip(axes.flat, metrics_to_plot, colors):
#     vals = perf_df[metric].values
#     bars = ax.bar(model_names, vals, color=color, alpha=0.85,
#                   width=bar_width, edgecolor="white")

#     ax.set_title(metric, fontsize=13, fontweight="bold")
#     y_min = min(0, min(vals) - 0.1)
#     ax.set_ylim(y_min, 1.15)
#     ax.set_ylabel(metric)
#     ax.set_xticks(range(len(model_names)))
#     ax.set_xticklabels(model_names, rotation=15, ha="right", fontsize=9)
#     ax.grid(axis="y", alpha=0.3)
#     ax.axhline(y=0, color='gray', linewidth=0.5, linestyle='--')
#     ax.spines["top"].set_visible(False)
#     ax.spines["right"].set_visible(False)

#     for bar, val in zip(bars, vals):
#         ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
#                 f"{val:.4f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

# plt.tight_layout()
# plt.savefig("model_comparison.png", dpi=150, bbox_inches="tight")
# print("Performance comparison chart saved as 'model_comparison.png'")
# plt.show()

# # # ===== VISUALIZATION: Regression Metrics Bar Chart =====
# # fig2, axes2 = plt.subplots(1, 3, figsize=(16, 5))
# # fig2.suptitle("Regression-Style Metrics (Binary Predictions vs Consensus Ground Truth)",
# #               fontsize=14, fontweight="bold")

# # reg_metrics = ["MSE", "MAE", "R2 Score"]
# # reg_colors = ["#D81B60", "#5E35B1", "#00897B"]

# # for ax, metric, color in zip(axes2, reg_metrics, reg_colors):
# #     vals = perf_df[metric].values
# #     bars = ax.bar(model_names, vals, color=color, alpha=0.85,
# #                   width=bar_width, edgecolor="white")

# #     ax.set_title(metric, fontsize=13, fontweight="bold")
# #     ax.set_ylabel(metric)
# #     ax.set_xticks(range(len(model_names)))
# #     ax.set_xticklabels(model_names, rotation=15, ha="right", fontsize=9)
# #     ax.grid(axis="y", alpha=0.3)
# #     ax.spines["top"].set_visible(False)
# #     ax.spines["right"].set_visible(False)

# #     for bar, val in zip(bars, vals):
# #         offset = 0.002 if val >= 0 else -0.015
# #         ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + offset,
# #                 f"{val:.4f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

# # plt.tight_layout()
# # plt.savefig("model_comparison_regression.png", dpi=150, bbox_inches="tight")
# # print("Regression metrics chart saved as 'model_comparison_regression.png'")
# # plt.show()


In [ ]:
# === VISUALIZATION: Classification Metrics Bar Chart ===
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("ML Model Performance Comparison (Classification Metrics)",
             fontsize=16, fontweight="bold")

metrics_to_plot = ["Precision", "Recall", "F1-Score", "Accuracy"]
colors = ["#E74C3C", "#2ECC71", "#3498DB", "#9B59B6"]
model_names = perf_df.index.tolist()

for ax, metric, color in zip(axes.flat, metrics_to_plot, colors):
    vals = perf_df[metric].values
    bars = ax.bar(model_names, vals, color=color, alpha=0.85,
                  width=0.5, edgecolor="white", linewidth=1.5)

    ax.set_title(metric, fontsize=14, fontweight="bold")
    y_min = min(0, min(vals) - 0.1)
    ax.set_ylim(y_min, 1.15)
    ax.set_ylabel(metric, fontsize=11)
    ax.set_xticks(range(len(model_names)))
    ax.set_xticklabels(model_names, rotation=15, ha="right", fontsize=10)
    ax.grid(axis="y", alpha=0.3)
    ax.axhline(y=0, color='gray', linewidth=0.5, linestyle='--')
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f"{val:.4f}", ha="center", va="bottom", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.savefig("model_comparison_classification.png", dpi=150, bbox_inches="tight")
print("Classification metrics chart saved as 'model_comparison_classification.png'")
plt.show()


In [ ]:
# === VISUALIZATION: Anomaly Predictions Time Series (Multi-line) ===
fig, ax = plt.subplots(1, 1, figsize=(18, 7))

rolling_window = 168

time_index = df_work.index
gt_rolling = pd.Series(consensus_gt, index=time_index).rolling(rolling_window, min_periods=1).mean()
ax.plot(time_index, gt_rolling, color='#1565C0', linewidth=2.0, label='Consensus Ground Truth', zorder=4)

model_colors = ['#FF9800', '#4CAF50', '#F44336']

for (name, preds), color in zip(models.items(), model_colors):
    p = preds.values.astype(int) if hasattr(preds, 'values') else preds.astype(int)
    rolling_avg = pd.Series(p, index=time_index).rolling(rolling_window, min_periods=1).mean()
    ax.plot(time_index, rolling_avg, color=color, linewidth=1.5, label=name, alpha=0.85)

ax.set_title('Anomaly Detection: Model Predictions Over Time (7-day Rolling Average)',
             fontsize=16, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Anomaly Rate', fontsize=12)
ax.legend(loc='upper right', fontsize=11, framealpha=0.9)
ax.grid(axis='both', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('anomaly_timeseries_comparison.png', dpi=150, bbox_inches='tight')
print("Time series comparison chart saved as 'anomaly_timeseries_comparison.png'")
plt.show()


In [ ]:
# Create master results with STRICTER logic
results = pd.DataFrame(index=df_work.index)

# Add individual building results
for col in df_work.columns:
    building_short = col.split('_')[-1]
    
    # NEW LOGIC: Must be flagged by AT LEAST 2 methods to count as anomaly
    # (Reduces false positives significantly)
    stat = stat_anomalies[f'{col}_stat']
    temp = temp_anomalies[f'{col}_temp']
    
    # Count how many methods flagged this point
    method_count = stat.astype(int) + temp.astype(int)
    
    # Only anomaly if 2+ methods agree, OR if ML says it's extreme
    results[f'{building_short}_anomaly'] = (method_count >= 2) | (
        ml_anomalies & (method_count >= 1)  # ML + any other method
    )

# Summary with realistic cost calculations
print("=== CORRECTED ANOMALY SUMMARY ===\n")

total_cost_impact = 0
cost_per_unit = 0.05  # $0.05 per unit

for col in df_work.columns:
    short_name = col.split('_')[-1]
    anomaly_count = results[f'{short_name}_anomaly'].sum()
    anomaly_rate = (anomaly_count / len(df_work)) * 100
    
    # More realistic cost calculation
    if anomaly_count > 0:
        # Only calculate cost for anomalies significantly above normal
        normal_median = df_work[col][~results[f'{short_name}_anomaly']].median()
        anomaly_values = df_work[col][results[f'{short_name}_anomaly']]
        
        # Excess above 2x normal median
        excess_per_event = (anomaly_values - 2 * normal_median).clip(lower=0).mean()
        total_excess = excess_per_event * anomaly_count * cost_per_unit
    else:
        total_excess = 0
    
    total_cost_impact += total_excess
    
    print(f"{short_name:12} | {anomaly_count:3} anomalies ({anomaly_rate:4.1f}%) | "
          f"Est. Cost: ${total_excess:,.0f}")

print(f"\n{'TOTAL':12} | Est. Annual Cost Impact: ${total_cost_impact:,.0f}")

# Health check
if total_cost_impact > 1000000:  # If over $1M, something is still wrong
    print("\n⚠️ WARNING: Cost still seems high. Check data units (are they in thousands?)")

In [ ]:
# Create the 6-panel dashboard
fig, axes = plt.subplots(3, 2, figsize=(18, 14))
fig.suptitle('Energy Anomaly Detection System - Analysis Dashboard', 
             fontsize=20, fontweight='bold', y=0.995)

# 1. Time series plot
ax = axes[0, 0]
for col in df_work.columns:
    short_name = col.split('_')[-1]
    ax.plot(df_work.index, df_work[col], label=short_name, alpha=0.7)
    
    # Mark anomalies
    anomalies = results[f'{short_name}_anomaly']
    if anomalies.any():
        ax.scatter(df_work.index[anomalies], df_work[col][anomalies], 
                  color='red', s=20, zorder=5)

ax.set_title('Consumption Timeline with Anomalies (Red Dots)', fontsize=12, fontweight='bold')
ax.set_ylabel('Consumption')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Anomaly heatmap (rolling 7-day count)
ax = axes[0, 1]
anomaly_matrix = []
labels = []

for col in df_work.columns:
    short_name = col.split('_')[-1]
    # Rolling 7-day anomaly count
    rolling_anomalies = results[f'{short_name}_anomaly'].rolling(168).sum().values
    anomaly_matrix.append(rolling_anomalies)
    labels.append(short_name)

im = ax.imshow(anomaly_matrix, aspect='auto', cmap='Reds', interpolation='nearest')
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels)
ax.set_title('Anomaly Intensity Heatmap (Weekly Rolling)', fontsize=12, fontweight='bold')
ax.set_xlabel('Time Index')
plt.colorbar(im, ax=ax, label='Anomaly Count')

# 3. Detailed view of most problematic building
ax = axes[1, 0]
# Find building with most anomalies
anomaly_counts = [results[f'{col.split("_")[-1]}_anomaly'].sum() for col in df_work.columns]
worst_idx = np.argmax(anomaly_counts)
worst_col = df_work.columns[worst_idx]
worst_short = worst_col.split('_')[-1]

ax.plot(df_work.index, df_work[worst_col], color='blue', alpha=0.6, label='Normal')
anomaly_idx = results[results[f'{worst_short}_anomaly']].index
if len(anomaly_idx) > 0:
    ax.scatter(anomaly_idx, df_work.loc[anomaly_idx, worst_col], 
              color='red', s=30, zorder=5, label='Anomaly', marker='x')

ax.set_title(f'{worst_short} - Most Anomalous Building', fontsize=12, fontweight='bold')
ax.set_ylabel('Consumption')
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Distribution comparison
ax = axes[1, 1]
# Compare normal vs anomalous distributions for worst building
normal_data = df_work[worst_col][~results[f'{worst_short}_anomaly']]
anomaly_data = df_work[worst_col][results[f'{worst_short}_anomaly']]

ax.hist(normal_data, bins=50, alpha=0.7, color='blue', label='Normal', density=True)
if len(anomaly_data) > 0:
    ax.hist(anomaly_data, bins=20, alpha=0.7, color='red', label='Anomalous', density=True)

ax.axvline(normal_data.mean(), color='blue', linestyle='--', linewidth=2, label='Normal Mean')
ax.set_title(f'{worst_short} - Distribution Analysis', fontsize=12, fontweight='bold')
ax.set_xlabel('Consumption')
ax.set_ylabel('Density')
ax.legend()

# 5. Cost impact by building
ax = axes[2, 0]
building_names = []
costs = []

for col in df_work.columns:
    short_name = col.split('_')[-1]
    anomaly_count = results[f'{short_name}_anomaly'].sum()
    
    normal_usage = df_work[col][~results[f'{short_name}_anomaly']].mean()
    anomaly_usage = df_work[col][results[f'{short_name}_anomaly']].mean()
    
    if not np.isnan(anomaly_usage):
        excess = (anomaly_usage - normal_usage) * anomaly_count * cost_per_unit
    else:
        excess = 0
    
    building_names.append(short_name)
    costs.append(max(0, excess))

colors = ['red' if c > 1000 else 'orange' if c > 500 else 'green' for c in costs]
bars = ax.bar(building_names, [c/1000 for c in costs], color=colors)
ax.set_title('Estimated Cost Impact of Anomalies', fontsize=12, fontweight='bold')
ax.set_ylabel('Excess Cost ($ thousands)')

for bar, val in zip(bars, costs):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'${val/1000:.1f}k', ha='center', va='bottom')

# 6. Anomaly rate by building type
ax = axes[2, 1]
type_stats = {'office': [], 'education': [], 'lodging': []}

for col in df_work.columns:
    parts = col.split('_')
    if len(parts) >= 2:
        btype = parts[1]
        short_name = col.split('_')[-1]
        rate = results[f'{short_name}_anomaly'].mean() * 100
        if btype in type_stats:
            type_stats[btype].append(rate)

avg_rates = [np.mean(type_stats[t]) if type_stats[t] else 0 for t in type_stats.keys()]
bars = ax.bar(type_stats.keys(), avg_rates, color=['#1E88E5', '#43A047', '#FDD835'])

ax.set_title('Anomaly Rate by Building Type', fontsize=12, fontweight='bold')
ax.set_ylabel('Anomaly Percentage (%)')

for bar, rate in zip(bars, avg_rates):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{rate:.1f}%', ha='center', va='bottom')

plt.tight_layout()
# Save to your current folder (wherever the notebook is)
plt.savefig('final_dashboard.png', dpi=150, bbox_inches='tight')
print("✅ Dashboard saved! Check your notebook folder for 'final_dashboard.png'")
plt.show()

In [ ]:
# Export cleaned data and results
df_work.to_csv('cleaned_energy_data.csv')
results.to_csv('anomaly_detection_results.csv')

# Create a professional summary report
summary_data = []

for col in df_work.columns:
    short_name = col.split('_')[-1]
    parts = col.split('_')
    building_type = parts[1] if len(parts) > 1 else 'unknown'
    
    anomaly_count = results[f'{short_name}_anomaly'].sum()
    anomaly_rate = (anomaly_count / len(df_work)) * 100
    health_score = max(0, 100 - anomaly_rate * 5)
    
    # Recalculate cost
    if anomaly_count > 0:
        normal_median = df_work[col][~results[f'{short_name}_anomaly']].median()
        anomaly_values = df_work[col][results[f'{short_name}_anomaly']]
        excess = (anomaly_values - 2 * normal_median).clip(lower=0).sum() * 0.05
    else:
        excess = 0
    
    summary_data.append({
        'Building_ID': col,
        'Short_Name': short_name,
        'Type': building_type,
        'Anomaly_Count': anomaly_count,
        'Anomaly_Rate_%': round(anomaly_rate, 2),
        'Health_Score': round(health_score, 1),
        'Cost_Impact_$': round(excess, 2),
        'Action_Required': 'Yes' if anomaly_count > 10 else 'Monitor' if anomaly_count > 0 else 'No'
    })

summary_df = pd.DataFrame(summary_data)
summary_df.to_csv('project_summary_report.csv', index=False)

print("✅ Exported 3 files:")
print("   1. cleaned_energy_data.csv - Processed time series data")
print("   2. anomaly_detection_results.csv - Boolean flags for anomalies")
print("   3. project_summary_report.csv - Executive summary table")
print("\nSummary Preview:")
print(summary_df.to_string(index=False))

In [ ]:
# Create the Streamlit app file - FIXED VERSION
app_code = """import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

st.set_page_config(page_title="EnergyGuard AI", page_icon="⚡", layout="wide")

st.markdown('''
<style>
    .main-header { font-size: 2.5rem; font-weight: bold; color: #1E88E5; }
    .metric-card { background-color: #f0f2f6; padding: 1rem; border-radius: 0.5rem; border-left: 4px solid #1E88E5; }
    .alert-box { background-color: #ffebee; padding: 1rem; border-radius: 0.5rem; border-left: 4px solid #f44336; }
</style>
''', unsafe_allow_html=True)

st.markdown('<p class="main-header">⚡ EnergyGuard AI</p>', unsafe_allow_html=True)
st.markdown("### Intelligent Anomaly Detection for Commercial Buildings")
st.markdown("---")

uploaded_file = st.file_uploader("📁 Upload your chilledwater.csv file", type=['csv'])

if uploaded_file is None:
    st.info("👆 Please upload your CSV file to begin analysis")
    st.stop()

@st.cache_data
def load_data(file):
    df = pd.read_csv(file, parse_dates=['timestamp'])
    return df

df = load_data(uploaded_file)

# Preprocessing
missing_pct = df.isnull().sum() / len(df)
cols_to_keep = missing_pct[missing_pct < 0.95].index
df_clean = df[cols_to_keep].copy()
building_cols = [c for c in df_clean.columns if c != 'timestamp']
completeness = df_clean[building_cols].count().sort_values(ascending=False)
buildings = completeness.head(5).index.tolist()

st.sidebar.header("⚙️ Control Panel")
selected_building = st.sidebar.selectbox(
    "Select Building", 
    buildings, 
    format_func=lambda x: x.replace('_', ' ').title()
)

cost_per_unit = st.sidebar.slider("Energy Cost ($/unit)", 0.01, 0.20, 0.05, 0.01)
sensitivity = st.sidebar.slider("Detection Sensitivity", 1, 10, 5)

if st.button("🚀 Run Anomaly Detection", type="primary"):
    with st.spinner("Analyzing building data... Please wait"):
        
        # Prepare data
        df_analysis = df_clean[['timestamp', selected_building]].copy()
        df_analysis = df_analysis.dropna()
        df_analysis.set_index('timestamp', inplace=True)
        df_analysis[selected_building] = df_analysis[selected_building].interpolate(method='time')
        
        # Feature engineering
        df_analysis['hour'] = df_analysis.index.hour
        df_analysis['day_of_week'] = df_analysis.index.dayofweek
        df_analysis['month'] = df_analysis.index.month
        df_analysis['rolling_mean_7d'] = df_analysis[selected_building].rolling(168, min_periods=24).mean()
        df_analysis['rolling_std_7d'] = df_analysis[selected_building].rolling(168, min_periods=24).std()
        df_analysis = df_analysis.dropna()
        
        # Detection 1: Rolling Z-Score
        z_scores = np.abs((df_analysis[selected_building] - df_analysis['rolling_mean_7d']) / df_analysis['rolling_std_7d'])
        df_analysis['z_anomaly'] = (z_scores > 3.5) & (df_analysis[selected_building] > df_analysis['rolling_mean_7d'] * 1.5)
        
        # Detection 2: Isolation Forest
        feature_cols = ['hour', 'day_of_week', 'month', 'rolling_mean_7d']
        X = StandardScaler().fit_transform(df_analysis[feature_cols])
        iso = IsolationForest(contamination=0.02 * sensitivity/5, random_state=42)
        df_analysis['ml_anomaly'] = iso.fit_predict(X) == -1
        
        # Detection 3: Seasonal baseline
        seasonal_med = df_analysis.groupby([df_analysis.index.month, df_analysis.index.hour])[selected_building].transform('median')
        deviation = np.abs(df_analysis[selected_building] - seasonal_med) / seasonal_med
        df_analysis['seasonal_anomaly'] = deviation > 3.0
        
        # Combine (2+ methods)
        method_count = df_analysis['z_anomaly'].astype(int) + df_analysis['seasonal_anomaly'].astype(int)
        df_analysis['is_anomaly'] = (method_count >= 2) | (df_analysis['ml_anomaly'] & (method_count >= 1))
        
        # Metrics
        total_anomalies = df_analysis['is_anomaly'].sum()
        anomaly_rate = (total_anomalies / len(df_analysis)) * 100
        
        if total_anomalies > 0:
            normal_median = df_analysis[~df_analysis['is_anomaly']][selected_building].median()
            anomaly_vals = df_analysis[df_analysis['is_anomaly']][selected_building]
            cost = ((anomaly_vals - 2*normal_median).clip(lower=0).sum() * cost_per_unit)
        else:
            cost = 0
        
        health_score = max(0, 100 - anomaly_rate * 5)
        
        # Display metrics
        col1, col2, col3, col4 = st.columns(4)
        
        with col1:
            st.markdown('<div class="metric-card">', unsafe_allow_html=True)
            st.metric("🚨 Anomalies", int(total_anomalies))
            st.markdown('</div>', unsafe_allow_html=True)
            
        with col2:
            st.markdown('<div class="metric-card">', unsafe_allow_html=True)
            st.metric("📊 Anomaly Rate", f"{anomaly_rate:.2f}%")
            st.markdown('</div>', unsafe_allow_html=True)
            
        with col3:
            st.markdown('<div class="metric-card">', unsafe_allow_html=True)
            st.metric("💚 Health Score", f"{health_score:.0f}/100")
            st.markdown('</div>', unsafe_allow_html=True)
            
        with col4:
            st.markdown('<div class="metric-card">', unsafe_allow_html=True)
            st.metric("💰 Cost Impact", f"${cost:,.0f}")
            st.markdown('</div>', unsafe_allow_html=True)
        
        # Visualization
        st.subheader("📈 Consumption Timeline")
        
        fig = go.Figure()
        normal_data = df_analysis[~df_analysis['is_anomaly']]
        anomaly_data = df_analysis[df_analysis['is_anomaly']]
        
        fig.add_trace(go.Scatter(
            x=normal_data.index, 
            y=normal_data[selected_building],
            mode='lines',
            name='Normal Usage',
            line=dict(color='#1E88E5', width=1),
            opacity=0.7
        ))
        
        if len(anomaly_data) > 0:
            fig.add_trace(go.Scatter(
                x=anomaly_data.index,
                y=anomaly_data[selected_building],
                mode='markers',
                name='Anomaly',
                marker=dict(color='#f44336', size=10, symbol='x', line=dict(width=2)),
                hovertemplate='Date: %{x}<br>Usage: %{y}<br>Type: Anomaly'
            ))
        
        fig.update_layout(
            title=f"{selected_building.replace('_', ' ').title()} - Energy Consumption Analysis",
            xaxis_title="Date",
            yaxis_title="Consumption",
            hovermode='x unified',
            height=500,
            showlegend=True
        )
        
        st.plotly_chart(fig, use_container_width=True)
        
        # Anomaly list
        if total_anomalies > 0:
            st.subheader("📋 Detected Anomalies")
            anomaly_table = anomaly_data[[selected_building]].copy()
            anomaly_table['Severity'] = ['High' if v > normal_median * 3 else 'Medium' 
                                        for v in anomaly_data[selected_building]]
            anomaly_table.columns = ['Consumption', 'Severity']
            st.dataframe(anomaly_table.sort_index(ascending=False), use_container_width=True)
            
            csv = anomaly_table.to_csv().encode('utf-8')
            st.download_button(
                label="📥 Download Anomaly Report (CSV)",
                data=csv,
                file_name=f'anomalies_{selected_building}.csv'
            )
        else:
            st.success("✅ No anomalies detected! Building is operating normally.")
        
        # Alert
        if anomaly_rate > 5:
            st.markdown(
                "<div class='alert-box'><strong>⚠️ High Anomaly Alert!</strong><br>" +
                f"{selected_building.replace('_', ' ').title()} shows {anomaly_rate:.1f}% anomaly rate. " +
                "Recommend immediate HVAC inspection.</div>", 
                unsafe_allow_html=True
            )

st.sidebar.markdown("---")
st.sidebar.info(\"\"\"
How to use:
1. Upload chilledwater.csv
2. Select a building
3. Adjust sensitivity if needed
4. Click 'Run Anomaly Detection'
5. Download reports
\"\"\")
"""

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(app_code)

print("Created 'app.py' - Your interactive web app is ready!")
print("\nTo run it, open a NEW terminal/command prompt and type:")
print("   streamlit run app.py")

In [ ]:
import os
print("Your current folder path is:")
print(os.getcwd())